# <p style="text-align: center;"> Legal Entity Identifier (LEI) Notebooks </p>
### <p style="text-align: center;">Name and Address Transformation for ISO 20022</p>


#### Tasks covered in this notebook:
- Download latest Golden Copy file
- Transforms LEI name and address into the ISO 20022-compliant hybrid structure.
- Validate if the primary legal name and address are in basic Latin (ASCII) script. If not, use a suitable translation or transliteration that is limited to ASCII characters, if it is available.
- If no basic Latin information exists for both name and address, the script returns the message "No data using basic Latin (ASCII) characters is available."
- ISO 20022 hybrid format supports 2 address lines of up to 70 characters each. The notebook combines this into a maximum of 140 characters. If the address exceeds this limit, it is truncated and marked with a "+" to indicate continuation.
- Shows how the same transformation and validation can be performed via the GLEIF API.

Note: This notebook does not alter or generate any address or name information. Its sole purpose is to restructure available LEI reference data to be used in ISO20022.

For more details, please refer to the blog: https://www.gleif.org/en/newsroom/blog/transforming-data-into-opportunities-metric-in-motion-unlocking-trusted-entity-data-for-cross-border-payments

In [1]:
# This cell determines whether the notebook is run in Google Colab.

try:
    import google.colab
    IN_COLAB = True
    
    # Connect your Google Drive to the Colab environment:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust the path:
    import sys
    sys.path.append('/content/drive/MyDrive/lei_notebooks') # Adjust this path, if necessary

except ImportError:
    IN_COLAB = False


In [2]:
# =============================================================================
# CONFIGURATION OPTIONS
# =============================================================================

# DATA SCOPE OPTIONS
USE_FULL_DATASET = False  # Set to 'False' to use only essential data elements as defined below. 'True' will use all data elements.

# STORAGE OPTIONS
SAVE_TO_DISK = False  # Set to 'False' to keep data only in memory. 'True' will store LEI data and mappings locally

# DISPLAY CONFIGURATION
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Data Scope: {'Full Dataset' if USE_FULL_DATASET else 'Essential Columns Only'}")
print(f"Storage: {'Save to Disk' if SAVE_TO_DISK else 'Keep in Memory Only'}")

print("=" * 60)


CONFIGURATION SUMMARY
Data Scope: Essential Columns Only
Storage: Keep in Memory Only


In [38]:
# Import required libraries and utils
from datetime import datetime
import pandas as pd
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field

# Import from utils (this will automatically run environment setup)
from utils import (
    GoldenCopyDownload,
    GLEIFAPI,
    ColumnNames,
    NameAddressResultVisualizer,
    TextXml
)

IN_COLAB = globals().get("IN_COLAB", False)

# Only use data element that are relevant for Address Mapping
REQUIRED_COLUMNS = ColumnNames.ESSENTIAL_COLUMNS + ColumnNames.OTHER_NAMES_COLUMNS + ColumnNames.TRANSLITERATED_OTHER_NAMES_COLUMNS + ColumnNames.LEGAL_ADDRESS_COLUMNS + ColumnNames.OTHER_ADDRESS_COLUMNS + ColumnNames.TRANSLITERATED_OTHER_ADDRESS_COLUMNS

if not USE_FULL_DATASET:
    print(f"Selected Columns: {len(REQUIRED_COLUMNS)} columns")

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

Selected Columns: 98 columns
Environment: Local


In [5]:
date = datetime.today().strftime("%Y-%m-%d")

# Alternatively, use specific date and time of day:

date = '2026-04-01' 
# time_preference = "00:00" # Golden Copy is available daily at 00:00, 08:00, 16:00 UTC

# Base data directory (relative to current working directory in notebook)
DEFAULT_DATA_DIR = Path.cwd() 

# Subfolder for GC downloads
DEFAULT_GC_DIR = DEFAULT_DATA_DIR / "gc_downloads"

# Subfolder for mappings 
DEFAULT_SAVE_DIR = DEFAULT_DATA_DIR / "downloads"

# Download data using the selected setup
level_1_data = await GoldenCopyDownload.download_with_config(
    date=date,
    save_to_disk=SAVE_TO_DISK,
    use_full_dataset=USE_FULL_DATASET,
    essential_columns=REQUIRED_COLUMNS if not USE_FULL_DATASET else None,
    save_dir=DEFAULT_GC_DIR,
    #time=time_preference,
)

Using subset of 98 columns
Found URL for: https://goldencopy.gleif.org/api/v2/golden-copies/publishes/lei2/20260401-0000.csv
Reading only selected columns: 98 columns
Successfully loaded 3,267,543 rows and 98 columns into memory
Data loaded successfully into memory
Data loaded in memory: 3,267,543 rows × 98 columns
Memory usage: 11095.6 MB


In [6]:
# Sneak peek at LEI data:
level_1_data.head(3)

,LEI,Entity.LegalName,Entity.LegalName.xmllang,Entity.OtherEntityNames.OtherEntityName.1,Entity.OtherEntityNames.OtherEntityName.1.xmllang,Entity.OtherEntityNames.OtherEntityName.1.type,Entity.OtherEntityNames.OtherEntityName.2,Entity.OtherEntityNames.OtherEntityName.2.xmllang,Entity.OtherEntityNames.OtherEntityName.2.type,Entity.OtherEntityNames.OtherEntityName.3,...,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.AddressNumberWithinBuilding,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.MailRouting,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.AdditionalAddressLine.1,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.AdditionalAddressLine.2,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.AdditionalAddressLine.3,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.City,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.Region,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.Country,Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress.2.PostalCode,Registration.RegistrationStatus
0,001GPB6A9XPE8XJICC14,Fidelity Advisor Leveraged Company Stock Fund,en,FIDELITY ADVISOR SERIES I - Fidelity Advisor L...,en,PREVIOUS_LEGAL_NAME,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LAPSED
1,004L5FPTUREIWK9T2N63,"Hutchin Hill Capital, LP",en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LAPSED
2,00EHHQ2ZHDCFXJCPCL46,Vanguard Fiduciary Trust Company Vanguard Russ...,en,VANGUARD FIDUCIARY TRUST COMPANY RUSSELL 1000 ...,en,PREVIOUS_LEGAL_NAME,Vanguard Russell 1000 Growth Index Trust,en,PREVIOUS_LEGAL_NAME,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ISSUED


### Mapping of LEI data to ISO 20022 compliant name and address data

The following methodology is applied to determine whether the primary, translated or transliterated LEI data is used for ISO 20022:
- The code first evaluates the primary (legal) name and address by checking the actual content to determine if they are Latin and ASCII-compliant.
- If the primary values are not ASCII (even if they are Latin), the code then checks the translated values (if available) for Latin/ASCII compliance.
- If no suitable translation is found, the code falls back to the transliterated values.
- If there is not Latin information available, the code returns the NO_INFO_MESSAGE "No data using basic Latin (ASCII) characters is available."

#### Source categories

Data is grouped into three categories:
- **primary**: primary LEI data (legal name or legal address)  
- **translation**: conversion of a data element from one language to another while preserving its meaning
- **transliteration**: phonetic conversion of text from one script to another to allow pronunciation, without translating the meaning

In [7]:
NO_INFO_MESSAGE = "No data using basic Latin (ASCII) characters is available."

# Translation alternatives for name and address
NAME_TRANSLATION_TYPE = "ALTERNATIVE_LANGUAGE_LEGAL_NAME"
ADDRESS_TRANSLATION_TYPES = "ALTERNATIVE_LANGUAGE_LEGAL_ADDRESS"

SOURCE_ORDER = ("primary", "translation", "transliteration")

HYBRID_ADR_LINE_MAX_LEN = 70
HYBRID_ADR_LINE_COUNT = 2

# columns to be displayed as output
VISIBLE_COLUMNS = [
    "LEI",
    "Legal Name",
    "Translated Name",
    "Transliterated Name",
    "Legal Address",
    "Translated Address",
    "Transliterated Address",
    "Source XML",
    "ISO 20022 Compliant Representation",
]

HELPER_COLUMNS = [
    "selected_name_source",
    "selected_address_source",
]

SOURCE_COLUMN_TO_TARGET_MAP = {
    "selected_name_source": {
        "primary": "Legal Name",
        "translation": "Translated Name",
        "transliteration": "Transliterated Name",
    },
    "selected_address_source": {
        "primary": "Legal Address",
        "translation": "Translated Address",
        "transliteration": "Transliterated Address",
    },
}

The below data classes define the structure for organizing and storing LEI data throughout the transformation process.
- NameValue and AddressValue represent individual name and address records along with their metadata (e.g., language, source, type).
- StructuredRecord groups all extracted names and addresses by their source (e.g., legal name/address, translated, transliterated).
- ConversionResult stores the final output, including all the name and address for ISO 20022 hybrid mapping, along with all available variants (legal name/address, translated, transliterated) for reference.

In [8]:
@dataclass
class NameValue:
    value: Optional[str]
    language: Optional[str]
    source: str
    name_type: Optional[str] = None


@dataclass
class AddressValue:
    language: Optional[str]
    source: str
    address_type: Optional[str]
    first_address_line: Optional[str]
    address_number: Optional[str]
    address_number_within_building: Optional[str]
    mail_routing: Optional[str]
    additional_address_lines: List[str]
    city: Optional[str]
    region: Optional[str]
    country: Optional[str]
    postal_code: Optional[str]


@dataclass
class StructuredRecord:
    lei: Optional[str]
    names_by_source: Dict[str, List[NameValue]] = field(default_factory=dict)
    addresses_by_source: Dict[str, List[AddressValue]] = field(default_factory=dict)


@dataclass
class ConversionResult:
    lei: Optional[str]
    source_xml: Optional[str]
    hybrid_xml: Optional[str]
    selected_name_source: Optional[str]
    selected_address_source: Optional[str]
    
    legal_name: Optional[str] = None
    translated_name: Optional[str] = None
    transliterated_name: Optional[str] = None

    legal_address: Optional[str] = None
    translated_address: Optional[str] = None
    transliterated_address: Optional[str] = None


The below section provides helper methods for cleaning, restructuring, and preparing text values for processing and XML generation. It handles formatting values for hybrid ISO 20022 output.

In [9]:
XML_NAMESPACES = {
    "lei": "urn:gleif:specification:lei:xsd:leidata:1.0",
    "xml": "http://www.w3.org/XML/1998/namespace",
}

def iso20022_text(value: Optional[str]) -> Optional[str]:
    """
    Transform text for ISO 20022 output while preserving user-visible content.

    The function trims whitespace and replaces selected unicode punctuation
    variants with stable ASCII equivalents used in XML output.
    """
    value = TextXml.clean(value)
    if not value:
        return None

    replacements = {
        "\u00A0": " ",    # non-breaking space
        "\u2013": "-",    # en dash
        "\u2014": "-",    # em dash
        "\u2018": "'",    # left single quote
        "\u2019": "'",    # right single quote
        "\u201C": '"',    # left double quote
        "\u201D": '"',    # right double quote
        "\u2026": "...",  # ellipsis
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    return value

def pretty_xml_fragment(xml_text: Optional[str]) -> Optional[str]:
    return TextXml.pretty_xml(xml_text, wrapper_namespaces=XML_NAMESPACES)

def add_iso20022_xml_text(
    parent: ET.Element,
    tag: str,
    value: Optional[str],
    max_length: Optional[int] = None,) -> None:
    TextXml.add_xml_text(
        parent=parent,
        tag=tag,
        value=value,
        max_length=max_length,
        transform=iso20022_text,
    )

def split_adr_lines(
    text: Optional[str],
    max_length: int = HYBRID_ADR_LINE_MAX_LEN,) -> tuple[Optional[str], Optional[str]]:
    """
    Split address text into up to two hybrid `AdrLine` values.

    Rules:
        - Maximum length per line is `max_length`.
        - `+` marks continuation when truncation occurs.
        - Boundary characters replaced by `+` are carried forward so letters are
          not lost because of continuation markers.

    Returns:
        Tuple of `(line_1, line_2)` where each item may be `None`.
    """
    text = iso20022_text(text)
    if not text:
        return None, None

    if len(text) <= max_length:
        return text, None

    first_chunk = text[:max_length]
    remaining = text[max_length:]

    # Preserve the boundary character replaced by '+' in line 1.
    line_1 = first_chunk[:-1] + "+"
    remaining = first_chunk[-1] + remaining

    if len(remaining) <= max_length:
        return line_1, remaining

    second_chunk = remaining[:max_length]
    tail = remaining[max_length:]
    # Preserve the boundary character replaced by '+' in line 2.
    line_2 = second_chunk[:-1] + "+" if tail else second_chunk

    if line_2 == line_1:
        line_2 = None

    return line_1, line_2

### Addresses

The HybridAddressConverter transforms LEI records into ISO 20022 hybrid XML format. It selects the appropriate name and address based on defined rules for translation and transliteration (see above). Furthermore, it generates both structured source XML and hybrid XML output. The following rules are applied to map the LEI address information to ISO 20022 compliant address data:

lei:country --> Ctry<br>
lei:city --> TwnNm<br>
lei:PostalCode --> PstCd<br>
lei:Region --> CtrySubDvsn<br>
lei:AddressNumber --> BldgNb<br>
FirstAddressLine + AdditionalAddressLine + All remaining unmapped address elements + MailRouting + AddressNumber --> AdrLine<br>

In [10]:
class HybridAddressConverter:
    """
    Convert LEI records into structured source XML and ISO 20022 hybrid XML.

    This class centralizes selection rules for name/address source priority and keeps
    rendering logic in one place so parsers can focus only on data extraction.
    """

    NAME_XML_MAP = {
        "primary": ("lei:LegalName", None),
        "translation": ("lei:OtherEntityName", "lei:OtherEntityNames"),
        "transliteration": (
            "lei:TransliteratedOtherEntityName",
            "lei:TransliteratedOtherEntityNames",
        ),
    }

    ADDRESS_XML_MAP = {
        "primary": ("lei:LegalAddress", None),
        "translation": ("lei:OtherAddress", "lei:OtherAddresses"),
        "transliteration": (
            "lei:TransliteratedOtherAddress",
            "lei:TransliteratedOtherAddresses",
        ),
    }

    # ------------- Public conversion methods -----------------
    
    def convert_record(self, record: StructuredRecord) -> ConversionResult:
        """
        Convert one record into mapping output.

        Args:
            record: Parsed LEI record grouped by source buckets.

        Returns:
            ConversionResult containing selected source metadata plus source and
            hybrid XML snippets.
        """
        chosen_name = self._choose_name(record.names_by_source)
        chosen_address = self._choose_address(record.addresses_by_source)
        source_xml = pretty_xml_fragment(
                self._render_source_xml(chosen_name, chosen_address)
            )
        hybrid_xml = pretty_xml_fragment(
            self._render_hybrid_xml(chosen_name, chosen_address)
        )
        # Always show an explicit fallback message instead of blank cells.
        source_xml = source_xml or NO_INFO_MESSAGE
        hybrid_xml = hybrid_xml or NO_INFO_MESSAGE

        # Add remark when only one ISO 20022 element is present
        name_present = chosen_name is not None
        address_present = chosen_address is not None
        if name_present ^ address_present and hybrid_xml != NO_INFO_MESSAGE:
            if name_present:
                remark = "<!-- Note: only name (Nm) is available; address (PstlAdr) is missing. -->"
            else:
                remark = "<!-- Note: only address (PstlAdr) is available; name (Nm) is missing. -->"
            hybrid_xml = f"{hybrid_xml}\n{remark}"

        return ConversionResult(
            lei=record.lei,
            source_xml=source_xml,
            hybrid_xml=hybrid_xml,
            selected_name_source=chosen_name.source if chosen_name else None,
            selected_address_source=chosen_address.source if chosen_address else None,
            legal_name=self._first_name_value(record.names_by_source.get("primary", [])),
            translated_name=self._first_name_value(record.names_by_source.get("translation", [])),
            transliterated_name=self._first_name_value(record.names_by_source.get("transliteration", [])),
            legal_address=self._first_address_value(record.addresses_by_source.get("primary", [])),
            translated_address=self._first_address_value(record.addresses_by_source.get("translation", [])),
            transliterated_address=self._first_address_value(record.addresses_by_source.get("transliteration", [])),
        )

    def convert_records(self, records: Iterable[StructuredRecord]) -> pd.DataFrame:
        """
        Build a DataFrame from ConversionResult objects.
        """
        return self._results_to_dataframe(
            self.convert_record(record) for record in records
        )

    def _results_to_dataframe(self, results: Iterable[ConversionResult]) -> pd.DataFrame:
        """
        Build a DataFrame from ConversionResult objects with interested columns.
        """
        rows = []
        for result in results:
            rows.append(
                {
                    "LEI": result.lei,
                    "selected_name_source": result.selected_name_source,
                    "selected_address_source": result.selected_address_source,
                    "Legal Name": result.legal_name,
                    "Translated Name": result.translated_name,
                    "Transliterated Name": result.transliterated_name,
                    "Legal Address": result.legal_address,
                    "Translated Address": result.translated_address,
                    "Transliterated Address": result.transliterated_address,
                    "Source XML": result.source_xml,
                    "ISO 20022 Compliant Representation": result.hybrid_xml,
                }
            )

        return pd.DataFrame(
            rows,
            columns=[
                "LEI",
                "selected_name_source",
                "selected_address_source",
                "Legal Name",
                "Translated Name",
                "Transliterated Name",
                "Legal Address",
                "Translated Address",
                "Transliterated Address",
                "Source XML",
                "ISO 20022 Compliant Representation",
            ],
        )

    # ------------- Shared selection methods -----------------
        
    def _address_text(self, address: AddressValue) -> str:
        parts = [
            address.first_address_line,
            address.address_number,
            address.address_number_within_building,
            address.mail_routing,
            *address.additional_address_lines,
            address.city,
            address.region,
            address.country,
            address.postal_code,
        ]
        return " ".join(part for part in parts if part)

    def _is_ascii_name(self, item: NameValue) -> bool:
        """
        Check ASCII compliance using the name text.
        """
        if not item.value:
            return False
        return TextXml.is_ascii_text(item.value)
    
    def _is_ascii_address(self, item: AddressValue) -> bool:
        """
        Check ASCII compliance using the address text.
        """
        text = self._address_text(item)
        if not text:
            return False
        return TextXml.is_ascii_text(text)

    # ------------- Name handling -----------------
    
    def _choose_name(
        self, names_by_source: Dict[str, List[NameValue]]
    ) -> Optional[NameValue]:
        """
        Pick the first eligible name based on selection order.

        Selection order:
            1. Primary legal name (ASCII-compatible)
            2. Translation (ASCII-compatible)
            3. Transliteration (ASCII-compatible)

        Args:
            names_by_source: Name candidates grouped by source key.

        Returns:
            Selected NameValue, or None when no candidate is eligible.
        """
        # 1. Primary / Legal Name, only if ASCII
        for item in names_by_source.get("primary", []):
             if self._is_ascii_name(item):
                return item

        # 2. Translation
        for item in names_by_source.get("translation", []):
            if self._is_ascii_name(item):
                return item
        
        for item in names_by_source.get("transliteration", []):
            if self._is_ascii_name(item):
                return item

        return None

    def _first_name_value(self, names: List[NameValue]) -> Optional[str]:
        """
        Return the first available name value for display/output.
        """
        for name in names:
            if name.value:
                return name.value
        return None

    def _append_name_xml(self, parent: ET.Element, name: NameValue) -> None:
        """
        Append a name element into the given XML parent node.
        """
        item_tag, container_tag = self.NAME_XML_MAP[name.source]

        if container_tag:
            container = ET.SubElement(parent, container_tag)
            element = ET.SubElement(container, item_tag)
            if name.source == "translation" and name.name_type:
                element.set("type", name.name_type)
            elif name.name_type:
                element.set("type", name.name_type)
        else:
            element = ET.SubElement(parent, item_tag)

        if name.language:
            element.set(
                "{http://www.w3.org/XML/1998/namespace}lang",
                name.language.lower(),
            )

        element.text = iso20022_text(name.value)

    def _render_name_source_xml(self, name: Optional[NameValue]) -> List[ET.Element]:
        """
        Render name source XML as element list.
        """
        if not name:
            return []

        root = ET.Element("tmp")
        self._append_name_xml(root, name)
        return list(root)

    def _render_name_hybrid_xml(self, name: Optional[NameValue]) -> List[ET.Element]:
        """
        Render hybrid XML name fragment.
        """
        if not name:
            return []

        root = ET.Element("tmp")
        add_iso20022_xml_text(root, "Nm", name.value, max_length=140)
        return list(root)

    # ------------- Address handling -----------------
    
    def _choose_address(
        self, addresses_by_source: Dict[str, List[AddressValue]]) -> Optional[AddressValue]:
        """
        Pick the first eligible address based on selection order.

        Selection order:
            1. Primary legal address (ASCII-compatible)
            2. Translation with ALTERNATIVE_LANGUAGE_LEGAL_ADDRESS type
            3. Transliteration (ASCII-compatible)

        Args:
            addresses_by_source: Address candidates grouped by source key.

        Returns:
            Selected AddressValue, or None when no candidate is eligible.
        """
    
        # 1. Primary / Legal Address
        for item in addresses_by_source.get("primary", []):
            if self._is_ascii_address(item):
                return item
    
        # 2. Translation
        for item in addresses_by_source.get("translation", []):
            if (
                self._is_ascii_address(item)
                and item.address_type == "ALTERNATIVE_LANGUAGE_LEGAL_ADDRESS"
            ):
                return item
    
        # 3. Transliteration
        for item in addresses_by_source.get("transliteration", []):
            if self._is_ascii_address(item):
                return item
    
        return None

    def _first_address_value(self, addresses: List[AddressValue]) -> Optional[str]:
        """
        Return the first available address value as a single display string.
        """
        for address in addresses:
            parts = [
                address.first_address_line,
                address.address_number,
                address.address_number_within_building,
                address.mail_routing,
                *address.additional_address_lines,
                address.city,
                address.region,
                address.country,
                address.postal_code,
            ]
            parts = [part for part in parts if part]
            if parts:
                return ", ".join(parts)
        return None

    def _append_address_xml(self, parent: ET.Element, address: AddressValue) -> None:
        """
        Append address elements into the given XML parent node.
        """
        item_tag, container_tag = self.ADDRESS_XML_MAP[address.source]

        if container_tag:
            container = ET.SubElement(parent, container_tag)
            element = ET.SubElement(container, item_tag)
            if address.address_type:
                element.set("type", address.address_type.lower())
        else:
            element = ET.SubElement(parent, item_tag)

        if address.language:
            element.set(
                "{http://www.w3.org/XML/1998/namespace}lang",
                address.language.lower(),
            )

        for tag, value in (
            ("lei:FirstAddressLine", address.first_address_line),
            ("lei:AddressNumber", address.address_number),
            ("lei:AddressNumberWithinBuilding", address.address_number_within_building),
            ("lei:MailRouting", address.mail_routing),
        ):
            add_iso20022_xml_text(element, tag, value)

        for line in address.additional_address_lines:
            add_iso20022_xml_text(element, "lei:AdditionalAddressLine", line)

        for tag, value in (
            ("lei:City", address.city),
            ("lei:Region", address.region),
            ("lei:Country", address.country),
            ("lei:PostalCode", address.postal_code),
        ):
            add_iso20022_xml_text(element, tag, value)

    def _render_address_source_xml(
        self, address: Optional[AddressValue]
    ) -> List[ET.Element]:
        """
        Render address source XML as element list.
        """
        if not address:
            return []

        root = ET.Element("tmp")
        self._append_address_xml(root, address)
        return list(root)

    def _build_hybrid_address_lines(
        self,
        address: AddressValue,
        extra_excluded: Optional[Sequence[Optional[str]]] = None,
    ) -> tuple[Optional[str], Optional[str]]:
        """
        Build up to two `AdrLine` values (max 70 chars each).

        Address parts are deduplicated and filtered against address fields
        already represented by dedicated hybrid tags.

        Args:
            address: Selected address payload.
            extra_excluded: Additional values to exclude from dedupe input.

        Returns:
            Tuple of (line_1, line_2), each possibly None.
        """
        extra_excluded = extra_excluded or []
        excluded_for_dedupe = [
            address.city,
            address.region,
            address.country,
            address.postal_code,
            *extra_excluded,
        ]

        parts = TextXml.deduplicate(
            values=[
                address.first_address_line,
                address.address_number,
                address.address_number_within_building,
                address.mail_routing,
                *address.additional_address_lines,
            ],
            excluded=excluded_for_dedupe,
        )

        combined = ",".join(parts) if parts else None
        return split_adr_lines(
            combined,
            max_length=HYBRID_ADR_LINE_MAX_LEN,
        )

    def _render_address_hybrid_xml(
        self,
        address: Optional[AddressValue],
        chosen_name: Optional[NameValue] = None,
    ) -> List[ET.Element]:
        """
        Render hybrid XML address snippet.
        """
        if not address:
            return []

        pstl_adr = ET.Element("PstlAdr")

        for tag, value in (
            ("TwnNm", address.city),
            ("Ctry", address.country),
            ("PstCd", address.postal_code),
            ("CtrySubDvsn", address.region),
            ("BldgNb", address.address_number),
        ):
            add_iso20022_xml_text(pstl_adr, tag, value)

        line_1, line_2 = self._build_hybrid_address_lines(
            address,
            extra_excluded=[chosen_name.value if chosen_name else None],
        )
        add_iso20022_xml_text(pstl_adr, "AdrLine", line_1)
        add_iso20022_xml_text(pstl_adr, "AdrLine", line_2)

        return [pstl_adr]

    # ------------- Combined XML rendering -----------------
    
    def _render_source_xml(
        self,
        name: Optional[NameValue],
        address: Optional[AddressValue],
    ) -> Optional[str]:
        """
        Render the selected structured fields into an XML snippet.
        """
        elements: List[ET.Element] = []
        elements.extend(self._render_name_source_xml(name))
        elements.extend(self._render_address_source_xml(address))

        if not elements:
            return None

        return "".join(
            ET.tostring(element, encoding="unicode") for element in elements
        )

    def _render_hybrid_xml(
        self,
        name: Optional[NameValue],
        address: Optional[AddressValue],
    ) -> str:
        """
        Render the selected fields into the ISO 20022 Compliant Representation.
        """
        elements: List[ET.Element] = []
        elements.extend(self._render_name_hybrid_xml(name))
        elements.extend(self._render_address_hybrid_xml(address, name))

        if not elements:
            return NO_INFO_MESSAGE

        return "".join(
            ET.tostring(element, encoding="unicode") for element in elements
        )

The below parser class has common helper methods for parsing raw LEI data into structured name and address objects. It ensures consistent creation, validation, and filtering of name and address values before they are added to the structured record.

In [11]:
class BaseLeiRecordParser:
    """
    Shared parsing helpers for converting raw LEI inputs into structured records.

    Concrete parsers provide source-specific extraction, while this base class
    handles cleanup, value construction, and acceptance rules.
    """
    
    # Record initialization
    def _empty_record(self, lei: Optional[str]) -> StructuredRecord:
        """
        Create an empty structured record with initialized source buckets.
        """
        return StructuredRecord(
            lei=lei,
            names_by_source={source: [] for source in SOURCE_ORDER},
            addresses_by_source={source: [] for source in SOURCE_ORDER},
        )

   # ---------- Name Helpers  ----------
    
    @staticmethod
    def _accept_name(source: str, name: NameValue) -> bool:
        """
        Return True when the extracted name should be kept for the given source.
        """
        if source == "translation":
            return name.name_type in NAME_TRANSLATION_TYPE
        return True

    def _make_name(
        self,
        value: Any,
        language: Any,
        source: str,
        name_type: Any,
    ) -> Optional[NameValue]:
        """
        Create a NameValue from raw fields, or None if empty.
        """
        value = TextXml.clean(value)
        if not value:
            return None

        return NameValue(
            value=value,
            language=TextXml.clean(language),
            source=source,
            name_type=TextXml.clean(name_type),
        )

   # ---------- Address Helpers  ----------
    
    @staticmethod
    def _accept_address(source: str, address: AddressValue) -> bool:
        """
        Return True when the extracted address should be kept for the given source.
        """
        if source == "translation":
            return address.address_type == ADDRESS_TRANSLATION_TYPES
        return True

    def _make_address(
        self,
        language: Any,
        source: str,
        address_type: Any,
        first_address_line: Any,
        address_number: Any,
        address_number_within_building: Any,
        mail_routing: Any,
        additional_address_lines: Iterable[Any],
        city: Any,
        region: Any,
        country: Any,
        postal_code: Any,
    ) -> Optional[AddressValue]:
        """
        Create an AddressValue from raw fields, or None if all fields are empty.
        """
        address = AddressValue(
            language=TextXml.clean(language),
            source=source,
            address_type=TextXml.clean(address_type),
            first_address_line=TextXml.clean(first_address_line),
            address_number=TextXml.clean(address_number),
            address_number_within_building=TextXml.clean(address_number_within_building),
            mail_routing=TextXml.clean(mail_routing),
            additional_address_lines=TextXml.unique(additional_address_lines),
            city=TextXml.clean(city),
            region=TextXml.clean(region),
            country=TextXml.clean(country),
            postal_code=TextXml.clean(postal_code),
        )

        if not any(
            [
                address.first_address_line,
                address.address_number,
                address.address_number_within_building,
                address.mail_routing,
                address.additional_address_lines,
                address.city,
                address.region,
                address.country,
                address.postal_code,
            ]
        ):
            return None

        return address

### Using the GoldenCopy

This section shows how to parse LEI Golden Copy data into structured records. It extracts relevant fields from the dataset and maps them into standardized name and address structures for further processing.

In [12]:
class GoldenCopyLeiRecordParser(BaseLeiRecordParser):
    """
    Parse flat Golden Copy rows into structured name/address source buckets.

    The parser maps numbered flat columns into structured `NameValue` and
    `AddressValue` objects that can be consumed by `HybridAddressConverter`.
    """

    def parse(self, row: Dict[str, Any]) -> StructuredRecord:
        """
        Transform a Golden Copy flat row into NameValue/AddressValue lists.
        """
        lei = TextXml.clean(row.get("LEI") or row.get("lei"))
        record = self._empty_record(lei)

        # Name selection
        original_name = self._make_name(
            value=row.get("Entity.LegalName"),
            language=row.get("Entity.LegalName.xmllang"),
            source="primary",
            name_type="LEGAL_NAME",
        )
        if original_name:
            record.names_by_source["primary"].append(original_name)

        record.names_by_source["translation"].extend(
            self._extract_flat_names(row, "Entity.OtherEntityNames.OtherEntityName", "translation")
        )
        record.names_by_source["transliteration"].extend(
            self._extract_flat_names(
                row, "Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName", "transliteration"
            )
        )

        # Address selection
        original_address = self._make_address(
            language=row.get("Entity.LegalAddress.xmllang"),
            source="primary",
            address_type="LEGAL_ADDRESS",
            first_address_line=row.get("Entity.LegalAddress.FirstAddressLine"),
            address_number=row.get("Entity.LegalAddress.AddressNumber"),
            address_number_within_building=row.get("Entity.LegalAddress.AddressNumberWithinBuilding"),
            mail_routing=row.get("Entity.LegalAddress.MailRouting"),
            additional_address_lines=[
                row.get("Entity.LegalAddress.AdditionalAddressLine.1"),
                row.get("Entity.LegalAddress.AdditionalAddressLine.2"),
                row.get("Entity.LegalAddress.AdditionalAddressLine.3"),
            ],
            city=row.get("Entity.LegalAddress.City"),
            region=row.get("Entity.LegalAddress.Region"),
            country=row.get("Entity.LegalAddress.Country"),
            postal_code=row.get("Entity.LegalAddress.PostalCode"),
        )
        if original_address:
            record.addresses_by_source["primary"].append(original_address)

        record.addresses_by_source["translation"].extend(
            self._extract_flat_addresses(row, "Entity.OtherAddresses.OtherAddress", "translation")
        )
        record.addresses_by_source["transliteration"].extend(
            self._extract_flat_addresses(
                row, "Entity.TransliteratedOtherAddresses.TransliteratedOtherAddress", "transliteration"
            )
        )

        return record

    def _extract_flat_names(self, row: Dict[str, Any], prefix: str, source: str) -> List[NameValue]:
        """
        Extract numbered other name columns for a single source (flat Golden Copy).
        """
        names: List[NameValue] = []
        for idx in range(1, 6):
            base = f"{prefix}.{idx}"
            name = self._make_name(
                value=row.get(base),
                language=row.get(f"{base}.xmllang"),
                source=source,
                name_type=row.get(f"{base}.type"),
            )
            if not name:
                continue
            if self._accept_name(source, name):
                names.append(name)
        return names

    def _extract_flat_addresses(self, row: Dict[str, Any], prefix: str, source: str) -> List[AddressValue]:
        """
        Extract numbered other-address columns for a single source (flat Golden Copy).
        """
        addresses: List[AddressValue] = []
        for idx in range(1, 3):
            base = f"{prefix}.{idx}"
            address = self._make_address(
                language=row.get(f"{base}.xmllang"),
                source=source,
                address_type=row.get(f"{base}.type"),
                first_address_line=row.get(f"{base}.FirstAddressLine"),
                address_number=row.get(f"{base}.AddressNumber"),
                address_number_within_building=row.get(f"{base}.AddressNumberWithinBuilding"),
                mail_routing=row.get(f"{base}.MailRouting"),
                additional_address_lines=[
                    row.get(f"{base}.AdditionalAddressLine.1"),
                    row.get(f"{base}.AdditionalAddressLine.2"),
                    row.get(f"{base}.AdditionalAddressLine.3"),
                ],
                city=row.get(f"{base}.City"),
                region=row.get(f"{base}.Region"),
                country=row.get(f"{base}.Country"),
                postal_code=row.get(f"{base}.PostalCode"),
            )
            if not address:
                continue
            if self._accept_address(source, address):
                addresses.append(address)
        return addresses

This code processes a subset of the GoldenCopy dataframe into the final output format. It parses each record, converts it into hybrid XML, and aggregates the results into a DataFrame for analysis, display, or export.

In [13]:
def convert_dataframe(
    df: pd.DataFrame,
    gc_parser: GoldenCopyLeiRecordParser,
    converter: HybridAddressConverter,
    chunk_size: Optional[int] = None,
    n_rows: Optional[int] = None,
    show_progress: bool = True,
) -> pd.DataFrame:
    """
    Convert Golden Copy rows into the LEI-to-ISO20022 mapping output.

    Args:
        df: Input Golden Copy dataframe.
        gc_parser: Parser that normalizes each flat row.
        converter: Converter that renders source XML and ISO 20022 output.
        chunk_size: Optional chunk size for memory-friendly processing.
        n_rows: Optional row cap for sampling/debug runs.
        show_progress: Whether to print chunk progress.

    Returns:
        DataFrame with selected source metadata and XML columns.
    """
    if n_rows is not None:
        df = df.iloc[:n_rows]

    total_rows = len(df)

    if chunk_size is None:
        records = (gc_parser.parse(row) for row in df.to_dict(orient="records"))
        return converter.convert_records(records)

    parts: List[pd.DataFrame] = []

    for start in range(0, total_rows, chunk_size):
        stop = min(start + chunk_size, total_rows)
        chunk = df.iloc[start:stop]

        records = (
            gc_parser.parse(row)
            for row in chunk.to_dict(orient="records")
        )
        result_chunk = converter.convert_records(records)
        parts.append(result_chunk)

        if show_progress:
            print(f"processed rows {start:,} to {stop:,} of {total_rows:,}")

    if not parts:
        return pd.DataFrame(
           columns=[
            "LEI",
            "selected_name_source",
            "selected_address_source",
            "Legal Name",
            "Translated Name",
            "Transliterated Name",
            "Legal Address",
            "Translated Address",
            "Transliterated Address",
            "Source XML",
            "ISO 20022 Compliant Representation",
        ]
    )

    return pd.concat(parts, ignore_index=True)


In [42]:
# Initialize Golden Copy Parser
gc_parser = GoldenCopyLeiRecordParser()

# Initialize Hybrid Address Parser
converter = HybridAddressConverter()

result_df = convert_dataframe(df=level_1_data.sample(n=10),
                              gc_parser=gc_parser,
                              converter=converter,
                              n_rows=10)

# Exclude rows where the ISO 20022 output only contains a partial element
note_token = "<!-- Note:"
result_df = result_df[
    ~result_df["ISO 20022 Compliant Representation"].astype(str).str.contains(
        note_token, na=False
    )
]

visualizer = NameAddressResultVisualizer(
    text_formatter= pretty_xml_fragment,
    special_text_columns=["Source XML", "ISO 20022 Compliant Representation"],
    special_text_passthrough_values=[NO_INFO_MESSAGE],
    highlight_fill_hex="DFFBC0",
    highlight_css="#DFFBC0",
    column_width_px=260,
    column_width_px_map={
        "LEI": 200,
        "Source XML": 299,
        "ISO 20022 Compliant Representation": 299,
    },
    column_bg_css_map={
        "ISO 20022 Compliant Representation": "#79D7C5",
    },
    column_width_excel=36,
)

print("The light grey background color highlights the data element used for ISO 20022. The turquoise background color highlights the final result: The ISO 20022-conforming output.")

visualizer.display_notebook(
    result_df,
    visible_columns=VISIBLE_COLUMNS,
    helper_columns=HELPER_COLUMNS,
    source_column_to_target_map=SOURCE_COLUMN_TO_TARGET_MAP,
    preformatted_columns=["Source XML", "ISO 20022 Compliant Representation"],
).hide(axis="index")

# Saving the output as excel
# visualizer.save_excel(
#     result_df,
#     output_file="lei_iso20022_output.xlsx",
#     visible_columns=VISIBLE_COLUMNS,
#     source_column_to_target_map=SOURCE_COLUMN_TO_TARGET_MAP,
# )


The light grey background color highlights the data element used for ISO 20022. The turquoise background color highlights the final result: The ISO 20022-conforming output.


LEI,Legal Name,Translated Name,Transliterated Name,Legal Address,Translated Address,Transliterated Address,Source XML,ISO 20022 Compliant Representation
9695001J688M11HKEY73,AC2E INVEST,None,None,"290 AV ROBESPIERRE, LA GARDE, FR, 83130",None,None,AC2E INVEST 290 AV ROBESPIERRE LA GARDE FR 83130,AC2E INVEST LA GARDE FR 83130 290 AV ROBESPIERRE
254900S10BIS0ETNAC77,CHATELAIN REAL ESTATE,None,None,"Kasteleinsplein 35, Elsene, BE, 1050",None,None,CHATELAIN REAL ESTATE Kasteleinsplein 35 Elsene BE 1050,CHATELAIN REAL ESTATE Elsene BE 1050 Kasteleinsplein 35
724500D0DDQTZAYWMY70,Start 96 B.V.,None,None,"Eric Kropstraat 106, Rotterdam, NL, 3071AE",None,None,Start 96 B.V. Eric Kropstraat 106 Rotterdam NL 3071AE,Start 96 B.V. Rotterdam NL 3071AE Eric Kropstraat 106
315700THU8MDS6FOE369,PRODVINALCO SA,None,None,"Calea BACIULUI, 2-4, CLUJ-NAPOCA, RO-CJ, RO, 400230",None,None,PRODVINALCO SA Calea BACIULUI 2-4 CLUJ-NAPOCA RO-CJ RO 400230,"PRODVINALCO SA CLUJ-NAPOCA RO 400230 RO-CJ 2-4 Calea BACIULUI,2-4"
549300X4FHWKKLEY5K67,"Ameriplex Superior Partners, LLC",None,None,"C/O John T Phair, 227 South Main Street, Suite 300, South Bend, US-IN, US, 46601",None,None,"Ameriplex Superior Partners, LLC C/O John T Phair 227 South Main Street Suite 300 South Bend US-IN US 46601","Ameriplex Superior Partners, LLC South Bend US 46601 US-IN C/O John T Phair,227 South Main Street,Suite 300"
724500NEDF4CTOGWS593,A.L. van Vliet Holding B.V.,None,None,"Krimkade 2b, Voorschoten, NL, 2251KA",None,None,A.L. van Vliet Holding B.V. Krimkade 2b Voorschoten NL 2251KA,A.L. van Vliet Holding B.V. Voorschoten NL 2251KA Krimkade 2b
549300F2MCYXVUYSME74,American Century Multiple Investment Trust - American Century U.S. Small Cap Value Equity Trust,None,None,"1 Freedom Valley Drive, Oaks, US-PA, US, 19456",None,None,American Century Multiple Investment Trust - American Century U.S. Small Cap Value Equity Trust 1 Freedom Valley Drive Oaks US-PA US 19456,American Century Multiple Investment Trust - American Century U.S. Small Cap Value Equity Trust Oaks US 19456 US-PA 1 Freedom Valley Drive
549300A0QPUDB5HG0S81,Ecco Heating Products Ltd.,None,None,"14310 - 111th Avenue, Suite 300 West, Edmonton, CA-AB, CA, T5S 1J7",None,None,Ecco Heating Products Ltd. 14310 - 111th Avenue Suite 300 West Edmonton CA-AB CA T5S 1J7,"Ecco Heating Products Ltd. Edmonton CA T5S 1J7 CA-AB 14310 - 111th Avenue,Suite 300 West"
3358005ZKNAVM9I9LT24,PARAM VALUE INVESTMENTS,None,None,"208/209 THE CAPITAL, PLOT NO. C-70, BLOCK G,, BANDRA KURLA COMPLEX, BANDRA EAST, MUMBAI, IN-MH, IN, 400051",None,None,"PARAM VALUE INVESTMENTS 208/209 THE CAPITAL, PLOT NO. C-70, BLOCK G, BANDRA KURLA COMPLEX, BANDRA EAST MUMBAI IN-MH IN 400051","PARAM VALUE INVESTMENTS MUMBAI IN 400051 IN-MH 208/209 THE CAPITAL, PLOT NO. C-70, BLOCK G,,BANDRA KURLA COMPLEX, BA+ NDRA EAST"
254900QOMG4H6IDNW033,Raven Lime Notch Mountain LLC,None,None,"c/o Kaliser & Associates PC, 17101 Preston Road, Suite 110, Dallas, US-TX, US, 75248",None,None,Raven Lime Notch Mountain LLC c/o Kaliser & Associates PC 17101 Preston Road Suite 110 Dallas US-TX US 75248,"Raven Lime Notch Mountain LLC Dallas US 75248 US-TX c/o Kaliser & Associates PC,17101 Preston Road,Suite 110"


### Using the GLEIF API

This section shows how to retrieve Name and Addresses from GLEIF API for specific LEIs.

First, let's initialize the GLEIF API client.

In [16]:
# Initialize API client
api_client = GLEIFAPI()

EXAMPLE_LEIS = [
    "ZX7KYZN3LRZ6GZGZ0D51",
    "254900NNA76LCRZQ9G72",
    "097900CAKA0000310860",
    "2138005TK8WH6MAH8D23",
    "2549005WL1W9GJ5TM121",
    "353800MTV7BD7U4PN952",
    "984500D6C7A9F6HBEB32",
    "2549008Y3ED8N4LMLO36",
    "529900RMFDO02HT7UD75",
    "254900ATUF6MF4YV6R78"
]

print(f"Example LEIs: {EXAMPLE_LEIS}")

Example LEIs: ['ZX7KYZN3LRZ6GZGZ0D51', '254900NNA76LCRZQ9G72', '097900CAKA0000310860', '2138005TK8WH6MAH8D23', '2549005WL1W9GJ5TM121', '353800MTV7BD7U4PN952', '984500D6C7A9F6HBEB32', '2549008Y3ED8N4LMLO36', '529900RMFDO02HT7UD75', '254900ATUF6MF4YV6R78']


This class fetches LEI data from the GLEIF API and transforms the nested API response into a structured record. It extracts legal, translated, and transliterated names and addresses, organizing them into a consistent format that can be used for further processing and XML conversion.

In [17]:
class ApiLeiRecordParser(BaseLeiRecordParser):
    """
    Parse GLEIF API payloads.

    The parser supports batch fetch/parse workflows and converts nested API
    structures into source-grouped `NameValue` and `AddressValue` lists.
    """
    
    def __init__(
        self,
        api_client: Any,
        leis: Optional[Sequence[str]] = None,
    ) -> None:
        self.api_client = api_client
        self.leis = list(leis or [])
        self.lei_data_dict: Dict[str, Dict[str, Any]] = {}

    # ---------- Internal helpers (data only) ----------
    
    def _fetch_data(self) -> None:
        """
        Populate self.lei_data_dict with API results per LEI.
        """
        self.lei_data_dict = {}

        for lei in self.leis:
            print(f"Fetching LEI data for LEI: {lei}")

            payload = self.api_client.fetch_lei_attrs(lei=lei)

            if not payload:
                self.lei_data_dict[lei] = {}
                continue

            self.lei_data_dict[lei] = payload

        print(f"\nData fetched for {len(self.lei_data_dict)} LEI(s)")

    def fetch_data(self) -> None:
        """
        Public wrapper for data fetch.
        """
        self._fetch_data()

    def parse_fetched_data(self) -> List[StructuredRecord]:
        """
        Parse fetched API responses into structured records.
        """
        records: List[StructuredRecord] = []

        for lei, payload in self.lei_data_dict.items():
            if not payload:
                records.append(self._empty_record(lei))
                continue
            records.append(self.parse(payload))

        return records

    def fetch_and_parse(self) -> List[StructuredRecord]:
        """
        Fetch API responses and parse them into structured records.
        """
        self.fetch_data()
        return self.parse_fetched_data()

    # ---------- Main parse method ----------
    
    def parse(self, payload: Dict[str, Any]) -> StructuredRecord:
        """
        Convert one GLEIF API payload into source-grouped values.

        Args:
            payload: API response attributes for a single LEI.

        Returns:
            StructuredRecord with populated name/address buckets.
        """
        lei = TextXml.clean(payload.get("lei") or payload.get("LEI"))
        entity = payload.get("entity", {})
        record = self._empty_record(lei)

        self._parse_primary_name(entity, record)
        self._parse_translated_names(entity, record)
        self._parse_transliterated_names(entity, record)

        self._parse_primary_address(entity, record)
        self._parse_translated_addresses(entity, record)
        self._parse_transliterated_addresses(entity, record)

        return record

    # ---------- Name parsing ----------
    
    def _parse_primary_name(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse the primary legal name from the API entity block.
        """
        legal_name = entity.get("legalName", {})

        primary_name = self._make_name(
            value=legal_name.get("name"),
            language=legal_name.get("language"),
            source="primary",
            name_type="LEGAL_NAME",
        )

        if primary_name:
            record.names_by_source["primary"].append(primary_name)

    def _parse_translated_names(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse translated names from the API entity block.
        """
        record.names_by_source["translation"].extend(
            self._extract_nested_names(
                items=entity.get("otherNames", []),
                source="translation",
            )
        )

    def _parse_transliterated_names(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse transliterated names from the API entity block.
        """
        record.names_by_source["transliteration"].extend(
            self._extract_nested_names(
                items=entity.get("transliteratedOtherNames", []),
                source="transliteration",
            )
        )

    def _extract_nested_names(
        self,
        items: Sequence[Dict[str, Any]],
        source: str,
    ) -> List[NameValue]:
        """
        Extract nested names for translation or transliteration.
        """
        result: List[NameValue] = []

        for item in items or []:
            name = self._make_name(
                value=item.get("name"),
                language=item.get("language"),
                source=source,
                name_type=item.get("type"),
            )

            if not name:
                continue

            if self._accept_name(source, name):
                result.append(name)

        return result

    # ---------- Address parsing ----------
    
    def _parse_primary_address(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse the primary legal address from the API entity block.
        """
        legal_address = entity.get("legalAddress", {})
        legal_lines = legal_address.get("addressLines") or []

        primary_address = self._make_address(
            language=legal_address.get("language"),
            source="primary",
            address_type="LEGAL_ADDRESS",
            first_address_line=legal_lines[0] if legal_lines else None,
            address_number=legal_address.get("addressNumber"),
            address_number_within_building=legal_address.get("addressNumberWithinBuilding"),
            mail_routing=legal_address.get("mailRouting"),
            additional_address_lines=legal_lines[1:],
            city=legal_address.get("city"),
            region=legal_address.get("region"),
            country=legal_address.get("country"),
            postal_code=legal_address.get("postalCode"),
        )

        if primary_address:
            record.addresses_by_source["primary"].append(primary_address)

    def _parse_translated_addresses(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse translated addresses from the API entity block.
        """
        record.addresses_by_source["translation"].extend(
            self._extract_nested_addresses(
                items=entity.get("otherAddresses", []),
                source="translation",
            )
        )

    def _parse_transliterated_addresses(
        self,
        entity: Dict[str, Any],
        record: StructuredRecord,
    ) -> None:
        """
        Parse transliterated addresses from the API entity block.
        """
        record.addresses_by_source["transliteration"].extend(
            self._extract_nested_addresses(
                items=entity.get("transliteratedOtherAddresses", []),
                source="transliteration",
            )
        )

    def _extract_nested_addresses(
        self,
        items: Sequence[Dict[str, Any]],
        source: str,
    ) -> List[AddressValue]:
        """
        Extract nested addresses.

        Translated addresses are ordered by type priority so
        `ALTERNATIVE_LANGUAGE_LEGAL_ADDRESS` is preferred when present.

        Args:
            items: Raw address objects from API payload.
            source: Source key (`translation` or `transliteration`).

        Returns:
            Cleaned list of accepted AddressValue records.
        """
        priority = {
            "ALTERNATIVE_LANGUAGE_LEGAL_ADDRESS": 0,
        }

        sorted_items = sorted(
            items or [],
            key=lambda x: priority.get((x or {}).get("type") or "", 99),
        )

        result: List[AddressValue] = []

        for item in sorted_items:
            lines = item.get("addressLines") or []

            address = self._make_address(
                language=item.get("language"),
                source=source,
                address_type=item.get("type"),
                first_address_line=lines[0] if lines else None,
                address_number=item.get("addressNumber"),
                address_number_within_building=item.get("addressNumberWithinBuilding"),
                mail_routing=item.get("mailRouting"),
                additional_address_lines=lines[1:],
                city=item.get("city"),
                region=item.get("region"),
                country=item.get("country"),
                postal_code=item.get("postalCode"),
            )

            if not address:
                continue

            if self._accept_address(source, address):
                result.append(address)

        return result

In [18]:
# Initialize the API parser
api_parser = ApiLeiRecordParser(
    api_client=api_client,
    leis=EXAMPLE_LEIS,
)
# Fetch data and parse the record
api_records = api_parser.fetch_and_parse()

# Convert the results
api_result_df = converter.convert_records(api_records)

print("The light grey background color highlights the data element used for ISO 20022. The turquoise background color highlights the final result: The ISO 20022-conforming output.")

# Visualize the results in notebook
visualizer.display_notebook(
    api_result_df,
    visible_columns=VISIBLE_COLUMNS,
    helper_columns=HELPER_COLUMNS,
    source_column_to_target_map=SOURCE_COLUMN_TO_TARGET_MAP,
    preformatted_columns=["Source XML", "ISO 20022 Compliant Representation"],
).hide(axis="index")


Fetching LEI data for LEI: ZX7KYZN3LRZ6GZGZ0D51
Fetching LEI data for LEI: 254900NNA76LCRZQ9G72
Fetching LEI data for LEI: 097900CAKA0000310860
Fetching LEI data for LEI: 2138005TK8WH6MAH8D23
Fetching LEI data for LEI: 2549005WL1W9GJ5TM121
Fetching LEI data for LEI: 353800MTV7BD7U4PN952
Fetching LEI data for LEI: 984500D6C7A9F6HBEB32
Fetching LEI data for LEI: 2549008Y3ED8N4LMLO36
Fetching LEI data for LEI: 529900RMFDO02HT7UD75
Fetching LEI data for LEI: 254900ATUF6MF4YV6R78

Data fetched for 10 LEI(s)
The light grey background color highlights the data element used for ISO 20022. The turquoise background color highlights the final result: The ISO 20022-conforming output.


LEI,Legal Name,Translated Name,Transliterated Name,Legal Address,Translated Address,Transliterated Address,Source XML,ISO 20022 Compliant Representation
ZX7KYZN3LRZ6GZGZ0D51,GE HEALTHCARE INTERNATIONAL JAPAN INVESTMENTS B.V.,None,None,"DE RONDOM 8, EINDHOVEN, NL-NB, NL, 5612 AP",None,None,GE HEALTHCARE INTERNATIONAL JAPAN INVESTMENTS B.V. DE RONDOM 8 EINDHOVEN NL-NB NL 5612 AP,GE HEALTHCARE INTERNATIONAL JAPAN INVESTMENTS B.V. EINDHOVEN NL 5612 AP NL-NB DE RONDOM 8
254900NNA76LCRZQ9G72,合同会社久保木ITエンジニアリング,None,Kuboki IT Engineering LLC,"東京都千代田区神田和泉町一丁目, ６番１６号ヤマトビル４０５, 千代田区, JP-13, JP, 101-0024","Yamato Building 405, 1-6-16 Kanda Izumicho, Chiyoda-ku, JP-13, JP, 101-0024",None,Kuboki IT Engineering LLC Yamato Building 405 1-6-16 Kanda Izumicho Chiyoda-ku JP-13 JP 101-0024,"Kuboki IT Engineering LLC Chiyoda-ku JP 101-0024 JP-13 Yamato Building 405,1-6-16 Kanda Izumicho"
097900CAKA0000310860,Blaško s.r.o.,None,Blasko s.r.o.,"Andreja Hlinku 2, Trnava, SK, 91701",None,None,Blasko s.r.o. Andreja Hlinku 2 Trnava SK 91701,Blasko s.r.o. Trnava SK 91701 Andreja Hlinku 2
2138005TK8WH6MAH8D23,디아지오코리아 주식회사,None,"DIAGEO KOREA CO., LTD.","932-94 DAEWOL-RO, DAEWOL-MYUN, ICHEON-SI, KR-41, KR, 17342",None,None,"DIAGEO KOREA CO., LTD. 932-94 DAEWOL-RO DAEWOL-MYUN ICHEON-SI KR-41 KR 17342","DIAGEO KOREA CO., LTD. ICHEON-SI KR 17342 KR-41 932-94 DAEWOL-RO,DAEWOL-MYUN"
2549005WL1W9GJ5TM121,Rentokil Initial Finance B.V.,None,None,"Oude Middenweg 77, Den Haag, NL-ZH, NL, 2491 AC",None,None,Rentokil Initial Finance B.V. Oude Middenweg 77 Den Haag NL-ZH NL 2491 AC,Rentokil Initial Finance B.V. Den Haag NL 2491 AC NL-ZH Oude Middenweg 77
353800MTV7BD7U4PN952,野村信託銀行株式会社/138481427,"The Nomura Trust and Banking Co., Ltd. /138481427",None,"大手町2-2-2, 東京都 千代田区, JP, 100-0004","2-2-2,Otemachi, Chiyoda-ku Tokyo, JP, 100-0004",None,"The Nomura Trust and Banking Co., Ltd. /138481427 2-2-2,Otemachi Chiyoda-ku Tokyo JP 100-0004","The Nomura Trust and Banking Co., Ltd. /138481427 Chiyoda-ku Tokyo JP 100-0004 2-2-2,Otemachi"
984500D6C7A9F6HBEB32,福州世海国际物流有限公司,"Fuzhou United Marine Logistics Co., Ltd.",None,"五四路158号环球广场7层10室, 鼓楼区, 福州市, CN-FJ, CN, 350001","Room 10, 7th Floor, Global Plaza, No. 158 Wusi Road, Gulou District, Fuzhou, CN-FJ, CN, 350001",None,"Fuzhou United Marine Logistics Co., Ltd. Room 10, 7th Floor, Global Plaza, No. 158 Wusi Road Gulou District Fuzhou CN-FJ CN 350001","Fuzhou United Marine Logistics Co., Ltd. Fuzhou CN 350001 CN-FJ Room 10, 7th Floor, Global Plaza, No. 158 Wusi Road,Gulou District"
2549008Y3ED8N4LMLO36,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,None,None,"Av 12 De Octubre, Lizardo Garcia, Quito, EC-P, EC, 170523",None,None,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA Av 12 De Octubre Lizardo Garcia Quito EC-P EC 170523,"PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA Quito EC 170523 EC-P Av 12 De Octubre,Lizardo Garcia"
529900RMFDO02HT7UD75,Hansa 23 Vermögen GmbH & Co. KG,None,Hansa 23 Vermogen GmbH & Co. KG,"Hansastr. 23, Lippstadt, DE-NW, DE, 59557",None,None,Hansa 23 Vermogen GmbH & Co. KG Hansastr. 23 Lippstadt DE-NW DE 59557,Hansa 23 Vermogen GmbH & Co. KG Lippstadt DE 59557 DE-NW Hansastr. 23
254900ATUF6MF4YV6R78,THE GREG & ROSIE LOCK FOUNDATION,None,None,"CROWN HOUSE, CROWN STREET, IPSWICH, GB, IP1 3HS",None,None,THE GREG & ROSIE LOCK FOUNDATION CROWN HOUSE CROWN STREET IPSWICH GB IP1 3HS,"THE GREG & ROSIE LOCK FOUNDATION IPSWICH GB IP1 3HS CROWN HOUSE,CROWN STREET"


Version 1.0.0 <br> 
Last updated: 9th April 2026